In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 8.5 MB/s eta 0:00:00


In [ ]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, names=columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
df[['Insulin','Glucose','BloodPressure','SkinThickness','BMI','DiabetesPedigreeFunction','Age']].replace(0, np.nan, inplace=True)

/tmp/ipython-input-3606440048.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[['Insulin','Glucose','BloodPressure','SkinThickness','BMI','DiabetesPedigreeFunction','Age']].replace(0, np.nan, inplace=True)


In [ ]:
df.fillna(df.mean(),inplace=True)

In [ ]:
df.isnull().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


In [ ]:
X=df.iloc[:,:-1]
y=df.iloc[:,-1]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 200)
    max_depth = trial.suggest_int("max_depth", 5, 20)

    model = RandomForestClassifier(
        n_estimators = n_estimators,
        max_depth = max_depth,
        random_state = 42
    )

    score = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy").mean()
    return score

In [ ]:
study = optuna.create_study(direction = 'maximize', sampler = optuna.samplers.TPESampler())
study.optimize(objective, n_trials=50)

[I 2025-10-29 15:27:23,956] A new study created in memory with name: no-name-4a085072-9e59-48dc-8e78-af13e5e6f148
[I 2025-10-29 15:27:25,686] Trial 0 finished with value: 0.7655071304811407 and parameters: {'n_estimators': 152, 'max_depth': 10}. Best is trial 0 with value: 0.7655071304811407.
[I 2025-10-29 15:27:26,721] Trial 1 finished with value: 0.7703985072637611 and parameters: {'n_estimators': 52, 'max_depth': 8}. Best is trial 1 with value: 0.7703985072637611.
[I 2025-10-29 15:27:32,546] Trial 2 finished with value: 0.7687325069972012 and parameters: {'n_estimators': 185, 'max_depth': 9}. Best is trial 1 with value: 0.7703985072637611.
[I 2025-10-29 15:27:35,396] Trial 3 finished with value: 0.7736505397840864 and parameters: {'n_estimators': 122, 'max_depth': 5}. Best is trial 3 with value: 0.7736505397840864.
[I 2025-10-29 15:27:37,962] Trial 4 finished with value: 0.7866853258696522 and parameters: {'n_estimators': 91, 'max_depth': 14}. Best is trial 4 with value: 0.786685325

In [ ]:
print(f'Best Trial Accuracy achieved: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best Trial Accuracy achieved: 0.786711981873917
Best hyperparameters: {'n_estimators': 61, 'max_depth': 13}


In [ ]:
from sklearn.metrics import accuracy_score

# Now using the best parameters determined using CV to get best results

final_model = RandomForestClassifier(**study.best_trial.params,random_state=42)
final_model.fit(X_train,y_train)

y_pred = final_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy of Testing using best params: {accuracy}')

Accuracy of Testing using best params: 0.7467532467532467


Using Different Samplers (GridSearch and RandomSearch)

In [ ]:
# Random search
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)

[I 2025-10-29 15:28:29,920] A new study created in memory with name: no-name-1e883187-52c6-4a4b-9fd6-aa6d1674bde7
[I 2025-10-29 15:28:31,477] Trial 0 finished with value: 0.7720245235239237 and parameters: {'n_estimators': 147, 'max_depth': 14}. Best is trial 0 with value: 0.7720245235239237.
[I 2025-10-29 15:28:33,768] Trial 1 finished with value: 0.7736638677862189 and parameters: {'n_estimators': 140, 'max_depth': 13}. Best is trial 1 with value: 0.7736638677862189.
[I 2025-10-29 15:28:35,196] Trial 2 finished with value: 0.7703718512594963 and parameters: {'n_estimators': 131, 'max_depth': 12}. Best is trial 1 with value: 0.7736638677862189.
[I 2025-10-29 15:28:37,155] Trial 3 finished with value: 0.7703585232573638 and parameters: {'n_estimators': 198, 'max_depth': 9}. Best is trial 1 with value: 0.7736638677862189.
[I 2025-10-29 15:28:38,711] Trial 4 finished with value: 0.7687458349993337 and parameters: {'n_estimators': 164, 'max_depth': 7}. Best is trial 1 with value: 0.773663

In [ ]:
from sklearn.metrics import accuracy_score

# Now using the best parameters determined using CV to get best results

final_model = RandomForestClassifier(**study.best_trial.params,random_state=42)
final_model.fit(X_train,y_train)

y_pred = final_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy of Testing using best params: {accuracy}')

Accuracy of Testing using best params: 0.7272727272727273


In [ ]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [ ]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective, n_trials=50)

[I 2025-10-29 15:29:40,365] A new study created in memory with name: no-name-00175ac5-4bb5-4467-bdad-64be5be165ae
[I 2025-10-29 15:29:41,301] Trial 0 finished with value: 0.7687458349993335 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7687458349993335.
[I 2025-10-29 15:29:42,821] Trial 1 finished with value: 0.7655071304811407 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7687458349993335.
[I 2025-10-29 15:29:43,347] Trial 2 finished with value: 0.7752765560442489 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7752765560442489.
[I 2025-10-29 15:29:44,377] Trial 3 finished with value: 0.7801812608290016 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 3 with value: 0.7801812608290016.
[I 2025-10-29 15:29:45,451] Trial 4 finished with value: 0.775303212048514 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 3 with value: 0.7801812

In [ ]:
from sklearn.metrics import accuracy_score

# Now using the best parameters determined using CV to get best results

final_model = RandomForestClassifier(**study.best_trial.params,random_state=42)
final_model.fit(X_train,y_train)

y_pred = final_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy of Testing using best params: {accuracy}')

Accuracy of Testing using best params: 0.7337662337662337


Visualisations using PLOTS

In [ ]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

Visualising the study of Bayesian CV

In [ ]:
plot_optimization_history(study).show()

In [ ]:
plot_parallel_coordinate(study).show()

In [ ]:
plot_slice(study).show()

In [ ]:
plot_contour(study).show()

In [ ]:
plot_param_importances(study).show()

HyperParameter Tuning with Multiple ML Models

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [ ]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200)

[I 2025-10-29 15:30:11,962] A new study created in memory with name: no-name-6f41a16e-5be8-406b-b796-86c7784d8dc5
[I 2025-10-29 15:30:13,351] Trial 0 finished with value: 0.7654471544715448 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 93, 'learning_rate': 0.05621855601844021, 'max_depth': 8, 'min_samples_split': 5, 'min_samples_leaf': 10}. Best is trial 0 with value: 0.7654471544715448.
[I 2025-10-29 15:30:16,372] Trial 1 finished with value: 0.760569105691057 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 146, 'learning_rate': 0.0927766030961277, 'max_depth': 11, 'min_samples_split': 10, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.7654471544715448.
[I 2025-10-29 15:30:16,925] Trial 2 finished with value: 0.7752351347042882 and parameters: {'classifier': 'RandomForest', 'n_estimators': 102, 'max_depth': 7, 'min_samples_split': 9, 'min_samples_leaf': 5, 'bootstrap': True}. Best is trial 2 with value: 0.7752351347042882.
[I 2025-10-29 1

In [ ]:
best_trial = study.best_trial
print(f'Best Trial Accuracy achieved: {best_trial.value}')
print(f'Best hyperparameters: {best_trial.params}')

Best Trial Accuracy achieved: 0.7915032679738562
Best hyperparameters: {'classifier': 'RandomForest', 'n_estimators': 65, 'max_depth': 20, 'min_samples_split': 9, 'min_samples_leaf': 3, 'bootstrap': False}


In [ ]:
studyDF=study.trials_dataframe()

In [ ]:
studyDF.head()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.765447,2025-10-29 15:30:11.964091,2025-10-29 15:30:13.351222,0 days 00:00:01.387131,NaN,NaN,GradientBoosting,NaN,NaN,0.056219,8.0,10.0,5.0,93.0,COMPLETE
1,1,0.760569,2025-10-29 15:30:13.352255,2025-10-29 15:30:16.372032,0 days 00:00:03.019777,NaN,NaN,GradientBoosting,NaN,NaN,0.092777,11.0,5.0,10.0,146.0,COMPLETE
2,2,0.775235,2025-10-29 15:30:16.372945,2025-10-29 15:30:16.925131,0 days 00:00:00.552186,NaN,True,RandomForest,NaN,NaN,NaN,7.0,5.0,9.0,102.0,COMPLETE
3,3,0.768723,2025-10-29 15:30:16.926002,2025-10-29 15:30:17.533922,0 days 00:00:00.607920,NaN,NaN,GradientBoosting,NaN,NaN,0.134653,3.0,1.0,3.0,99.0,COMPLETE
4,4,0.768675,2025-10-29 15:30:17.535127,2025-10-29 15:30:17.581257,0 days 00:00:00.046130,1.589625,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [ ]:
studyDF['params_classifier'].value_counts()

,count
params_classifier,
RandomForest,172
GradientBoosting,15
SVM,13


In [ ]:
studyDF['value'].groupby(studyDF['params_classifier']).mean()

,value
params_classifier,
GradientBoosting,0.761223
RandomForest,0.780705
SVM,0.735137
